# Evaluate the live agent

Scores one Fabric data agent against answers we already know, checked in SQL (`labs/01-data-agent/verify_answers.sql`).

**How it works.** `evaluate_data_agent` asks the agent every question in the table below. A second model, **the judge**, reads each
answer (the most recent message in the conversation) next to the expected answer and replies *Yes*, *No* or *Unclear*, following the **critic prompt** (strict about numbers, relaxed about wording).
The results land in two lakehouse tables: a summary and the step-by-step detail, including the SQL the agent wrote.

**Use it like a unit test.** Run it after every change to instructions, example queries or data. In Demo 1, run it on the blank
live agent (everything fails), paste the three pieces, and run it again (everything passes).

Attach this notebook to the **lh__spaceparts** lakehouse. The evaluation SDK is in preview.


## Settings
`STAGE = 'sandbox'` scores the **draft** you're editing in the agent's page (no need to publish). `'production'` scores the published version that apps call.
`REPEATS`: agents don't answer identically every time; ask each question more than once (e.g. 3) to measure consistency before you put a question on stage.


In [ ]:
AGENT = "SpaceParts Sales Analyst (live)"   # display name of the data agent to score
STAGE = "sandbox"                           # "sandbox" = draft, "production" = published
REPEATS = 1                                 # times each question is asked
TABLE = "live_agent_eval"                   # results table in the attached lakehouse

In [ ]:
%pip install -q fabric-data-agent-sdk

In [ ]:
import pandas as pd
from fabric.dataagent.evaluation import evaluate_data_agent, get_evaluation_summary, get_evaluation_details

## The test set
Questions the Sales agent should answer, each with the SQL-checked answer. Add your own: one row per question your users really ask.


In [ ]:
ground_truth = pd.DataFrame([
 {
  "question": "What were net sales in euros this year?",
  "expected_answer": "| Net sales (eur) | |---| | €9,026,084,019 |"
 },
 {
  "question": "What were net sales in euros by region this year?",
  "expected_answer": "| Interplanetary region | Net sales (eur) | |---|---| | Îlos V Region | €2,399,631,496 | | Îlos IV Region | €1,279,339,327 | | Îlos VI-VII Region | €985,838,728 | | Sol IV Region | €937,428,218 | | Lakonía II Region | €853,627,661 | | Sol III Region | €732,974,087 | | Lakonía III-VIII Region | €462,458,164 | | Transit Gate Region | €436,608,980 | | Sol Belt Region | €291,461,881 | | Lakonía IX Region | €289,570,918 | | Eliad V Region | €167,647,592 | | Arcadia III Region | €147,951,495 | | Sol V Region | €38,325,640 | | Lakonía Belt Region | €1,789,641 | | Sol VI-VIII Region | €1,430,191 |"
 },
 {
  "question": "Which key accounts are furthest behind budget this year?",
  "expected_answer": "| Key account name | Actual (eur) | Budget (eur) | Gap (eur) | Gap (%) | |---|---|---|---|---| | Government Contract | €1,168,869,267 | €1,273,433,656 | €-104,564,389 | -8.2% | | Supernova Spaceparts | €342,380,213 | €371,872,861 | €-29,492,648 | -7.9% | | Pioneering Systems | €346,142,027 | €363,579,984 | €-17,437,957 | -4.8% | | Black Flag Spaceships | €268,780,019 | €281,906,391 | €-13,126,372 | -4.7% | | Sigil Interstellar | €355,404,464 | €361,737,958 | €-6,333,494 | -1.8% |"
 },
 {
  "question": "How did net sales grow against last year by product type?",
  "expected_answer": "| Product type | Ytd 2022 (eur) | Same period 2021 (eur) | Growth (%) | |---|---|---|---| | Life Support | €846,776,246 | €728,336,646 | 16.3% | | Engine & Propulsion | €860,404,898 | €782,120,992 | 10.0% | | Structural | €796,802,027 | €737,031,442 | 8.1% | | Projectiles | €2,272,624,262 | €2,136,474,563 | 6.4% | | Modules | €722,281,661 | €690,401,122 | 4.6% | | Electrical | €358,204,551 | €349,077,289 | 2.6% | | Fuel & Piping | €675,466,887 | €680,650,585 | -0.8% | | Hull & Shields | €2,055,583,143 | €2,158,885,933 | -4.8% | | Planetary Systems | €437,940,345 | €529,017,433 | -17.2% |"
 },
 {
  "question": "Who are the top 5 account managers by net sales this year?",
  "expected_answer": "| Account manager | Net sales (eur) | |---|---| | Felix Kratochvílová | €290,099,471 | | Przemysław Bocca | €225,953,651 | | Sára Menendez | €213,126,925 | | Josh Blanc | €212,442,160 | | Františka Tyma | €200,820,204 |"
 },
 {
  "question": "What were net sales in euros by region in 2022?",
  "expected_answer": "| Interplanetary region | Net sales (eur) | |---|---| | Îlos V Region | €2,399,631,496 | | Îlos IV Region | €1,279,339,327 | | Îlos VI-VII Region | €985,838,728 | | Sol IV Region | €937,428,218 | | Lakonía II Region | €853,627,661 | | Sol III Region | €732,974,087 | | Lakonía III-VIII Region | €462,458,164 | | Transit Gate Region | €436,608,980 | | Sol Belt Region | €291,461,881 | | Lakonía IX Region | €289,570,918 | | Eliad V Region | €167,647,592 | | Arcadia III Region | €147,951,495 | | Sol V Region | €38,325,640 | | Lakonía Belt Region | €1,789,641 | | Sol VI-VIII Region | €1,430,191 |"
 }
])
ground_truth

## The judge's marking guide (critic prompt)
Lenient about formatting, strict about numbers: an answer passes only if every figure in the expected answer appears, rounded to the nearest euro or 0.1%, and the period and names match.
The SDK fills in `{query}` and `{expected_answer}`; the judge reads the agent's answer from the conversation, so don't add an `{actual_answer}` placeholder.


In [ ]:
critic_prompt = '''
You are grading a data agent. Compare the most recent answer in this conversation with the ground truth for the query.
Ignore formatting, headings, bullet points, extra explanation, the order of columns, currency symbols and thousands separators.
Every number in the ground truth must appear in the answer, allowing rounding to the nearest whole euro or 0.1 percentage point.
Names (regions, key accounts, managers, product types) must match.
If the answer uses a different period or currency, or adds up local currencies instead of euros, it is not equivalent.

Query: {query}

Ground Truth: {expected_answer}

Return only one word: Yes, No, or Unclear.
'''

## Run the evaluation
Each question takes 20–40 seconds.


In [ ]:
evaluation_id = evaluate_data_agent(
    ground_truth,
    AGENT,
    table_name=TABLE,
    data_agent_stage=STAGE,
    critic_prompt=critic_prompt,
    num_query_repeats=REPEATS,
)
print("evaluation id:", evaluation_id)

## Score


In [ ]:
summary = get_evaluation_summary(TABLE)
display(summary)

## Read every answer
For anything that isn't *Yes*, open the agent's steps: compare its SQL with `verify_answers.sql`, fix **one layer** (tables, instructions or an example query), and run this notebook again.
A wrong answer here is also a good source of a new **example query**: copy the SQL it should have written into the agent's example queries.


In [ ]:
details = get_evaluation_details(evaluation_id, TABLE, get_all_rows=True, verbose=True)